# 05 · SQL views and business queries (MySQL 8)

**Project:** BYD France – Business Intelligence & Market Analytics · **Phase 4, step 2**

### Purpose
Answer the business questions **in SQL**, on top of the validated star schema from Notebook 04, and prove the SQL results equal the Python results from Notebook 03.

| Script | What it does |
|---|---|
| `01`–`03` | Rebuild and reload the model (a new Colab session starts empty), then re-run the 22 load checks |
| `04_views.sql` | Six reusable **views**. Power BI will connect to these, so each business rule exists once |
| `06_view_checks.sql` | 28 checks: every headline figure from Notebook 03 recomputed from the views |
| `05_business_queries.sql` | 10 business queries (Q1, Q3, Q4, Q5) using CTEs, joins and window functions |

### SQL techniques used, and why
| Technique | Where | Why |
|---|---|---|
| **Views** | all of `04` | one definition of "share", "rank", "growth" for every report |
| **CTE** (`WITH`) | `v_brand_period`, Q4.3 | break a calculation into readable named steps |
| **`RANK() OVER`** | rankings | ties get the same rank; brands not listed have no row, so they are never ranked as 0 |
| **`SUM() OVER (ORDER BY …)`** | Q3.2 | running total, compared with CCFA's published YTD |
| **`LAG(…, 12)`** | market YoY | only on the market series, which has every month |
| **Self-join on `year - 1`** | brand growth | safer than `LAG` when a brand is missing in some periods: no prior figure gives **NULL growth**, not a comparison with the wrong year |
| **Conditional aggregation** | Q4.1, Chinese-owned views | pivot years into columns; count Chinese-owned brands |

### How to run
**Runtime → Run all.** When step 2 asks, select `brand_month.csv`, `market_month.csv` and `dim_brand.csv` from `data/processed/`.

## 1 · Install and start MySQL

In [ ]:
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!service mysql start
!mysql -e "SET GLOBAL local_infile = 1; SELECT VERSION() AS mysql_version, @@local_infile AS local_infile_enabled;"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld        su: warning: cannot change directory to /nonexistent: No such file or directory
[ OK ]
+-------------------------+----------------------+
| mysql_version           | local_infile_enabled |
+-------------------------+----------------------+
| 8.0.46-0ubuntu0.24.04.5 |                    1 |
+-------------------------+----------------------+


## 2 · Upload the validated CSVs and verify their fingerprints

In [ ]:
import hashlib, io, re, subprocess
import pandas as pd
from IPython.display import display, Markdown

EXPECTED_SHA256 = {
    "brand_month.csv":  "75c9a78bbe244ce9577894190f5f41fbb38daaf69011726edb2a895e745013ad",
    "market_month.csv": "7f0b6a204092b78e96f1cda41d8766e07545404c09196f4002dc9bea1367b017",
    "dim_brand.csv":    "d34bfc04a27d3038852513c1b00e26dcde4f1e479458ad87e9d386588de4d442",
}
try:
    from google.colab import files
    files.upload()
except ImportError:
    pass
for name, sha in EXPECTED_SHA256.items():
    ok = hashlib.sha256(open(name, "rb").read()).hexdigest() == sha
    print(f"{name:<18} matches validated v2: {ok}")
    assert ok, f"{name} is not the validated v2 file"

def mysql(sql_file=None, query=None, db="byd_france", local_infile=False):
    """Run a .sql file or a query with the mysql client; return the result tables as DataFrames."""
    cmd = ["mysql", "-B"] + (["--local-infile=1"] if local_infile else []) + ([db] if db else [])
    cmd += ["-e", query] if query else []
    res = subprocess.run(cmd, stdin=open(sql_file) if sql_file else None, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(res.stderr)
    blocks, cur = [], []
    for line in res.stdout.splitlines():
        if cur and line.split("\t")[0] in ("loaded_table", "id"):
            blocks.append(cur); cur = []
        cur.append(line)
    if cur: blocks.append(cur)
    return [pd.read_csv(io.StringIO("\n".join(b)), sep="\t", na_values=["NULL"], keep_default_na=False) for b in blocks]

brand_month.csv    matches validated v2: True
market_month.csv   matches validated v2: True
dim_brand.csv      matches validated v2: True


## 3 · Write the SQL scripts
Identical to the files in `sql/` on GitHub.

In [ ]:
%%writefile 01_schema.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 01_schema.sql            (MySQL 8.0.16+)
-- Star schema for the validated Phase 2 v2 datasets.
--
--   dim_brand  (51)     dim_date (34)
--        \               /
--      fact_brand_month (1 734 = 34 months x 51 brands)
--                 |
--      fact_market_month (34)   <- market totals, one row per month
--
-- Design rules
--   * Missing stays missing: registrations are NULL when CCFA did not list
--     the brand that month (value_type = 'not_listed'). Never 0.
--   * The market total lives only in fact_market_month (not repeated on
--     every brand row); join on date_key when a share is needed.
--   * CHECK constraints encode the Phase 2 rules, so bad rows are rejected.
-- Re-running this script drops and rebuilds everything.
-- =====================================================================

CREATE DATABASE IF NOT EXISTS byd_france
  CHARACTER SET utf8mb4 COLLATE utf8mb4_0900_ai_ci;
USE byd_france;

DROP TABLE IF EXISTS fact_brand_month;
DROP TABLE IF EXISTS fact_market_month;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_brand;

-- ---------------------------------------------------------------------
-- dim_brand: one row per brand label used in the CCFA tables
-- ---------------------------------------------------------------------
CREATE TABLE dim_brand (
  brand                      VARCHAR(40)  NOT NULL,
  brand_origin_country       VARCHAR(40)  NULL,      -- NULL only for AUTRES (residual group)
  controlling_owner          VARCHAR(80)  NULL,
  controlling_owner_country  VARCHAR(40)  NULL,
  ownership_structure        VARCHAR(40)  NULL,
  is_chinese_owned           BOOLEAN      NOT NULL,  -- controlling owner (>50% / sole control) HQ in China
  is_chinese_origin          BOOLEAN      NOT NULL,  -- brand heritage, separate from ownership
  is_joint_venture           BOOLEAN      NOT NULL,
  is_byd                     BOOLEAN      NOT NULL,
  classification_note        VARCHAR(255) NULL,
  classification_rule        VARCHAR(255) NOT NULL,
  PRIMARY KEY (brand)
);

-- ---------------------------------------------------------------------
-- dim_date: one row per reporting month (CCFA publishes monthly)
-- ---------------------------------------------------------------------
CREATE TABLE dim_date (
  date_key          DATE        NOT NULL,   -- first day of the month
  period            CHAR(7)     NOT NULL,   -- 'YYYY-MM'
  year              SMALLINT    NOT NULL,
  month             TINYINT     NOT NULL,
  month_name        CHAR(3)     NOT NULL,
  quarter           TINYINT     NOT NULL,
  months_in_data    TINYINT     NOT NULL,   -- months of that year present in the dataset
  is_complete_year  BOOLEAN     NOT NULL,   -- TRUE for 2024 and 2025; FALSE for 2026 (Jan-Oct)
  PRIMARY KEY (date_key),
  UNIQUE KEY uq_period (period),
  CONSTRAINT chk_month CHECK (month BETWEEN 1 AND 12),
  CONSTRAINT chk_first_day CHECK (DAY(date_key) = 1)
);

-- Generate Jan 2024 .. Oct 2026 with a recursive CTE (no hard-coded rows).
INSERT INTO dim_date (date_key, period, year, month, month_name, quarter, months_in_data, is_complete_year)
WITH RECURSIVE m AS (
  SELECT DATE('2024-01-01') AS d
  UNION ALL
  SELECT d + INTERVAL 1 MONTH FROM m WHERE d < '2026-10-01'
)
SELECT d,
       DATE_FORMAT(d, '%Y-%m'),
       YEAR(d),
       MONTH(d),
       DATE_FORMAT(d, '%b'),
       QUARTER(d),
       COUNT(*) OVER (PARTITION BY YEAR(d)),
       COUNT(*) OVER (PARTITION BY YEAR(d)) = 12
FROM m;

-- ---------------------------------------------------------------------
-- fact_market_month: CCFA total French new passenger-car registrations
-- ---------------------------------------------------------------------
CREATE TABLE fact_market_month (
  date_key                    DATE         NOT NULL,
  market_registrations_month  INT          NOT NULL,
  market_registrations_ytd    INT          NOT NULL,   -- CCFA published YTD (never re-summed)
  ytd_revision_gap            INT          NULL,       -- published YTD - (previous YTD + month)
  value_type                  VARCHAR(12)  NOT NULL,
  source_id                   VARCHAR(10)  NOT NULL,   -- -> docs/source_register.csv
  source_file                 VARCHAR(80)  NOT NULL,
  PRIMARY KEY (date_key),
  CONSTRAINT fk_mkt_date FOREIGN KEY (date_key) REFERENCES dim_date (date_key),
  CONSTRAINT chk_mkt_positive CHECK (market_registrations_month > 0 AND market_registrations_ytd > 0)
);

-- ---------------------------------------------------------------------
-- fact_brand_month: one row per brand per month (grain = date_key x brand)
-- ---------------------------------------------------------------------
CREATE TABLE fact_brand_month (
  date_key             DATE          NOT NULL,
  brand                VARCHAR(40)   NOT NULL,
  brand_label          VARCHAR(40)   NULL,       -- label exactly as printed by CCFA
  registrations_month  INT           NULL,       -- NULL = not listed (unknown), never 0
  registrations_ytd    INT           NULL,       -- CCFA published YTD
  ytd_revision_gap     INT           NULL,
  share_month_pct      DECIMAL(6,3)  NULL,       -- CCFA published share, %
  share_ytd_pct        DECIMAL(6,3)  NULL,
  value_type           VARCHAR(12)   NOT NULL,   -- least reliable of the two below
  value_type_month     VARCHAR(12)   NOT NULL,
  value_type_ytd       VARCHAR(12)   NOT NULL,
  listed_in_source     BOOLEAN       NOT NULL,
  in_market_sum        BOOLEAN       NOT NULL,   -- TRUE = part of the comparison universe
  source_id            VARCHAR(10)   NOT NULL,
  source_file          VARCHAR(80)   NOT NULL,
  note                 VARCHAR(255)  NULL,
  PRIMARY KEY (date_key, brand),
  KEY ix_brand (brand),
  KEY ix_universe (in_market_sum, date_key),
  CONSTRAINT fk_fbm_date  FOREIGN KEY (date_key) REFERENCES dim_date (date_key),
  CONSTRAINT fk_fbm_brand FOREIGN KEY (brand)    REFERENCES dim_brand (brand),
  CONSTRAINT chk_value_type CHECK (
        value_type       IN ('actual','aggregated','imputed','not_listed')
    AND value_type_month IN ('actual','aggregated','imputed','not_listed')
    AND value_type_ytd   IN ('actual','aggregated','imputed','not_listed')),
  -- missing stays missing: not_listed <=> no registration figures
  CONSTRAINT chk_not_listed_is_null CHECK (
        (value_type =  'not_listed' AND registrations_month IS NULL AND registrations_ytd IS NULL AND share_month_pct IS NULL)
     OR (value_type <> 'not_listed' AND registrations_month IS NOT NULL)),
  CONSTRAINT chk_universe_listed CHECK (in_market_sum = FALSE OR value_type <> 'not_listed'),
  CONSTRAINT chk_non_negative CHECK (registrations_month >= 0 AND registrations_ytd >= 0)
);


Writing 01_schema.sql


In [ ]:
%%writefile 02_load.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 02_load.sql              (MySQL 8.0)
-- Loads the three validated Phase 2 v2 CSVs (unchanged) into the schema.
--
-- Run from the folder that holds the CSVs, with LOCAL INFILE enabled:
--   mysql --local-infile=1 byd_france < 02_load.sql
--
-- Conversion rules
--   * Empty CSV field  -> NULL  (NULLIF(@x, '')). Never 0.
--   * '105.0' (pandas float export) -> 105 via DECIMAL, exact because every
--     count is a whole number (checked in Phase 2).
--   * 'True' / 'False' -> 1 / 0.
-- IMPORTANT: LOAD DATA LOCAL turns data errors into WARNINGS and skips the
-- bad row instead of failing. So each load is followed by a warning count,
-- and 03_checks.sql verifies every row count. Expected warnings: 0.
-- =====================================================================

USE byd_france;
SET FOREIGN_KEY_CHECKS = 1;

-- Remove previously loaded facts so the script can be re-run safely (dim_date is kept).
DELETE FROM fact_brand_month;
DELETE FROM fact_market_month;
DELETE FROM dim_brand;

-- ---------------------------------------------------------------- dim_brand
LOAD DATA LOCAL INFILE 'dim_brand.csv'
INTO TABLE dim_brand
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(brand, @origin, @owner, @owner_country, @structure,
 @cn_owned, @cn_origin, @jv, @is_byd, @note, classification_rule)
SET brand_origin_country      = NULLIF(@origin, ''),
    controlling_owner         = NULLIF(@owner, ''),
    controlling_owner_country = NULLIF(@owner_country, ''),
    ownership_structure       = NULLIF(@structure, ''),
    is_chinese_owned          = (@cn_owned  = 'True'),
    is_chinese_origin         = (@cn_origin = 'True'),
    is_joint_venture          = (@jv        = 'True'),
    is_byd                    = (@is_byd    = 'True'),
    classification_note       = NULLIF(@note, '');
SELECT 'dim_brand' AS loaded_table, ROW_COUNT() AS rows_loaded, @@warning_count AS warnings;

-- -------------------------------------------------------- fact_market_month
LOAD DATA LOCAL INFILE 'market_month.csv'
INTO TABLE fact_market_month
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(@period, @date, @year, @month, @mkt_m, @mkt_ytd, @gap, value_type, source_id, source_file)
SET date_key                   = @date,
    market_registrations_month = CAST(@mkt_m   AS DECIMAL(12,1)),
    market_registrations_ytd   = CAST(@mkt_ytd AS DECIMAL(12,1)),
    ytd_revision_gap           = CAST(NULLIF(@gap, '') AS DECIMAL(12,1));
SELECT 'fact_market_month' AS loaded_table, ROW_COUNT() AS rows_loaded, @@warning_count AS warnings;

-- --------------------------------------------------------- fact_brand_month
-- The CSV also carries period/year/month and the market totals; they are
-- read into variables and not stored (dim_date and fact_market_month hold them).
LOAD DATA LOCAL INFILE 'brand_month.csv'
INTO TABLE fact_brand_month
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(@period, @date, @year, @month, brand, @label, @reg_m, @reg_ytd, @gap,
 @mkt_m, @mkt_ytd, @sh_m, @sh_ytd, value_type, value_type_month, value_type_ytd,
 @listed, @in_sum, source_id, source_file, @note)
SET date_key            = @date,
    brand_label         = NULLIF(@label, ''),
    registrations_month = CAST(NULLIF(@reg_m,   '') AS DECIMAL(12,1)),
    registrations_ytd   = CAST(NULLIF(@reg_ytd, '') AS DECIMAL(12,1)),
    ytd_revision_gap    = CAST(NULLIF(@gap,     '') AS DECIMAL(12,1)),
    share_month_pct     = NULLIF(@sh_m,   ''),
    share_ytd_pct       = NULLIF(@sh_ytd, ''),
    listed_in_source    = (@listed = 'True'),
    in_market_sum       = (@in_sum = 'True'),
    note                = NULLIF(@note, '');
SELECT 'fact_brand_month' AS loaded_table, ROW_COUNT() AS rows_loaded, @@warning_count AS warnings;


Writing 02_load.sql


In [ ]:
%%writefile 03_checks.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 03_checks.sql            (MySQL 8.0)
-- Reconciliation checks after loading. Every row must return 'PASS'.
-- The expected values come from the validated Phase 2 files and the
-- Notebook 03 results, so passing proves SQL and Python agree.
--   mysql -t byd_france < 03_checks.sql
-- =====================================================================

USE byd_france;

WITH
universe AS (                               -- comparison universe: no Fiat/Abarth components, no not_listed rows
  SELECT * FROM fact_brand_month WHERE in_market_sum = 1
),
u_month AS (
  SELECT u.date_key,
         SUM(u.registrations_month) AS u_month,
         SUM(u.registrations_ytd)   AS u_ytd
  FROM universe u
  GROUP BY u.date_key
),
checks AS (
  SELECT 'K01' AS id, 'dim_brand rows' AS check_name, 51 AS expected,
         (SELECT COUNT(*) FROM dim_brand) AS actual
  UNION ALL SELECT 'K02', 'dim_date rows (Jan 2024 - Oct 2026)', 34, (SELECT COUNT(*) FROM dim_date)
  UNION ALL SELECT 'K03', 'fact_market_month rows', 34, (SELECT COUNT(*) FROM fact_market_month)
  UNION ALL SELECT 'K04', 'fact_brand_month rows (34 x 51)', 1734, (SELECT COUNT(*) FROM fact_brand_month)
  UNION ALL SELECT 'K05', 'months without exactly 51 brand rows', 0,
         (SELECT COUNT(*) FROM (SELECT date_key FROM fact_brand_month GROUP BY date_key HAVING COUNT(*) <> 51) x)
  UNION ALL SELECT 'K06', 'months where universe <> market total (month)', 0,
         (SELECT COUNT(*) FROM fact_market_month m LEFT JOIN u_month u ON u.date_key = m.date_key
          WHERE u.u_month IS NULL OR u.u_month <> m.market_registrations_month)
  UNION ALL SELECT 'K07', 'months where universe <> market total (YTD)', 0,
         (SELECT COUNT(*) FROM fact_market_month m LEFT JOIN u_month u ON u.date_key = m.date_key
          WHERE u.u_ytd IS NULL OR u.u_ytd <> m.market_registrations_ytd)
  UNION ALL SELECT 'K08', 'Fiat / Abarth component rows in universe', 0,
         (SELECT COUNT(*) FROM universe WHERE brand IN ('Fiat', 'Abarth'))
  UNION ALL SELECT 'K09', 'not_listed rows with any figure (zero-fill)', 0,
         (SELECT COUNT(*) FROM fact_brand_month WHERE value_type = 'not_listed'
          AND (registrations_month IS NOT NULL OR registrations_ytd IS NOT NULL OR share_month_pct IS NOT NULL))
  UNION ALL SELECT 'K10', 'not_listed rows in universe', 0,
         (SELECT COUNT(*) FROM universe WHERE value_type = 'not_listed')
  UNION ALL SELECT 'K11', 'NULL monthly registrations preserved (as in CSV)', 260,
         (SELECT COUNT(*) FROM fact_brand_month WHERE registrations_month IS NULL)
  UNION ALL SELECT 'K12', 'zero registrations (CCFA-published zeros only)', 6,
         (SELECT COUNT(*) FROM fact_brand_month WHERE registrations_month = 0)
  UNION ALL SELECT 'K13', 'universe rows without an ownership classification', 0,
         (SELECT COUNT(*) FROM universe u LEFT JOIN dim_brand b ON b.brand = u.brand WHERE b.brand IS NULL)
  UNION ALL SELECT 'K14', 'Chinese-owned brands in dim_brand', 12,
         (SELECT COUNT(*) FROM dim_brand WHERE is_chinese_owned = 1)
  UNION ALL SELECT 'K15', 'complete-year months in dim_date (2024 + 2025)', 24,
         (SELECT COUNT(*) FROM dim_date WHERE is_complete_year = 1)
  UNION ALL SELECT 'K16', 'BYD full year 2024 (Dec published YTD)', 5415,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2024-12-01')
  UNION ALL SELECT 'K17', 'BYD full year 2025 (Dec published YTD)', 13533,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2025-12-01')
  UNION ALL SELECT 'K18', 'BYD Jan-Oct 2025 (published YTD)', 9843,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2025-10-01')
  UNION ALL SELECT 'K19', 'BYD Jan-Oct 2026 (published YTD)', 28086,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2026-10-01')
  UNION ALL SELECT 'K20', 'Market full year 2025 (Dec published YTD)', 1632154,
         (SELECT market_registrations_ytd FROM fact_market_month WHERE date_key = '2025-12-01')
  UNION ALL SELECT 'K21', 'Market Jan-Oct 2026 (published YTD)', 1384293,
         (SELECT market_registrations_ytd FROM fact_market_month WHERE date_key = '2026-10-01')
  UNION ALL SELECT 'K22', 'Xpeng months with figures (listed from Mar 2025)', 20,
         (SELECT COUNT(registrations_month) FROM fact_brand_month WHERE brand = 'Xpeng')
)
SELECT id, check_name, expected, actual,
       CASE WHEN actual = expected THEN 'PASS' ELSE 'FAIL' END AS result
FROM checks
ORDER BY id;


Writing 03_checks.sql


In [ ]:
%%writefile 04_views.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 04_views.sql             (MySQL 8.0)
-- Reusable analytical views. Power BI connects to these, so every
-- business rule is defined once, in the database.
--
--   v_brand_month           comparison universe + calendar + market + ownership
--   v_market_month          market totals, same-month YoY, seasonality index
--   v_brand_period          full-year and Jan-Oct totals, share, rank, growth
--   v_brand_rank_month      monthly registration rank (AUTRES excluded)
--   v_chinese_owned_month   Chinese-owned share: lower bound + AUTRES upper bound
--   v_chinese_owned_period  same, per full year / Jan-Oct (published YTD)
--
-- Rules carried over from Phases 2-3
--   * Universe = in_market_sum = 1 (no Fiat/Abarth components, no not_listed rows).
--   * Period totals use CCFA published YTD at the period end (Dec or Oct).
--   * Growth compares the SAME period one year earlier through a join on
--     year - 1, never LAG(): if a brand is missing in the earlier period,
--     growth is NULL instead of silently comparing with an older year.
--   * Missing stays missing: no COALESCE(..., 0) on brand registrations.
-- Requires 01_schema.sql and 02_load.sql. Safe to re-run.
-- =====================================================================

USE byd_france;

-- ---------------------------------------------------------------------
-- v_brand_month: one row per brand per month, comparison universe only
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS v_brand_month;
CREATE VIEW v_brand_month AS
SELECT f.date_key,
       d.period,
       d.year,
       d.month,
       d.is_complete_year,
       f.brand,
       b.is_chinese_owned,
       b.is_chinese_origin,
       b.is_byd,
       CASE WHEN f.brand = 'Other brands (AUTRES)' THEN 1 ELSE 0 END AS is_residual,
       f.registrations_month,
       f.registrations_ytd,
       m.market_registrations_month,
       m.market_registrations_ytd,
       f.registrations_month * 100.0 / m.market_registrations_month AS share_month_pct,
       f.registrations_ytd   * 100.0 / m.market_registrations_ytd   AS share_ytd_pct,
       f.value_type,
       f.source_id
FROM fact_brand_month  f
JOIN dim_date          d ON d.date_key = f.date_key
JOIN fact_market_month m ON m.date_key = f.date_key
JOIN dim_brand         b ON b.brand    = f.brand
WHERE f.in_market_sum = 1;

-- ---------------------------------------------------------------------
-- v_market_month: market totals with same-month YoY and seasonality
-- (the market is present every month, so LAG(..., 12) is safe here)
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS v_market_month;
CREATE VIEW v_market_month AS
SELECT d.date_key,
       d.period,
       d.year,
       d.month,
       d.is_complete_year,
       m.market_registrations_month,
       m.market_registrations_ytd,
       LAG(m.market_registrations_month, 12) OVER (ORDER BY d.date_key) AS market_same_month_prev_year,
       m.market_registrations_month * 100.0
         / LAG(m.market_registrations_month, 12) OVER (ORDER BY d.date_key) - 100 AS market_yoy_pct,
       CASE WHEN d.is_complete_year = 1
            THEN m.market_registrations_month * 100.0
                 / AVG(m.market_registrations_month) OVER (PARTITION BY d.year)
       END AS seasonality_index      -- complete years only; 100 = average month
FROM fact_market_month m
JOIN dim_date d ON d.date_key = m.date_key;

-- ---------------------------------------------------------------------
-- v_brand_period: full years (Dec YTD) and like-for-like Jan-Oct (Oct YTD)
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS v_brand_period;
CREATE VIEW v_brand_period AS
WITH period_end AS (
  SELECT CASE WHEN v.month = 12 THEN 'full_year' ELSE 'jan_oct' END AS period_type,
         v.year,
         v.brand,
         v.is_chinese_owned,
         v.is_chinese_origin,
         v.is_byd,
         v.is_residual,
         v.registrations_ytd         AS registrations,
         v.market_registrations_ytd  AS market_registrations
  FROM v_brand_month v
  WHERE (v.month = 12 AND v.is_complete_year = 1)
     OR  v.month = 10
)
SELECT c.period_type,
       CONCAT(CASE c.period_type WHEN 'full_year' THEN 'FY ' ELSE 'Jan-Oct ' END, c.year) AS period_label,
       c.year,
       c.brand,
       c.is_chinese_owned,
       c.is_chinese_origin,
       c.is_byd,
       c.is_residual,
       c.registrations,
       c.market_registrations,
       c.registrations * 100.0 / c.market_registrations                    AS share_pct,
       p.registrations                                                     AS registrations_prev_year,
       c.registrations * 100.0 / NULLIF(p.registrations, 0) - 100          AS growth_pct,  -- NULL if no prior figure
       CASE WHEN c.is_residual = 0
            THEN RANK() OVER (PARTITION BY c.period_type, c.year, c.is_residual ORDER BY c.registrations DESC)
       END                                                                 AS rank_in_period,
       CASE WHEN c.is_residual = 0
            THEN COUNT(*) OVER (PARTITION BY c.period_type, c.year, c.is_residual)
       END                                                                 AS brands_ranked
FROM period_end c
LEFT JOIN period_end p
       ON p.period_type = c.period_type
      AND p.brand       = c.brand
      AND p.year        = c.year - 1;

-- ---------------------------------------------------------------------
-- v_brand_rank_month: monthly rank among listed brands (AUTRES excluded)
-- A brand not listed that month has no row, so it is never ranked as 0.
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS v_brand_rank_month;
CREATE VIEW v_brand_rank_month AS
SELECT v.date_key,
       v.period,
       v.year,
       v.month,
       v.brand,
       v.registrations_month,
       v.share_month_pct,
       RANK()   OVER (PARTITION BY v.date_key ORDER BY v.registrations_month DESC) AS rank_month,
       COUNT(*) OVER (PARTITION BY v.date_key)                                     AS brands_ranked
FROM v_brand_month v
WHERE v.is_residual = 0;

-- ---------------------------------------------------------------------
-- v_chinese_owned_month: lower bound (listed Chinese-owned brands) and
-- upper bound (+ AUTRES, which may hide unlisted Chinese brands)
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS v_chinese_owned_month;
CREATE VIEW v_chinese_owned_month AS
SELECT a.date_key,
       a.period,
       a.year,
       a.month,
       a.cn_brands_listed,
       a.cn_registrations,
       a.byd_registrations,
       a.autres_registrations,
       a.market_registrations,
       a.cn_registrations * 100.0 / a.market_registrations                           AS cn_share_lower_pct,
       a.autres_registrations * 100.0 / a.market_registrations                       AS autres_share_pct,
       (a.cn_registrations + a.autres_registrations) * 100.0 / a.market_registrations AS cn_share_upper_pct,
       a.byd_registrations * 100.0 / NULLIF(a.cn_registrations, 0)                   AS byd_part_of_cn_pct
FROM (
  SELECT v.date_key, v.period, v.year, v.month,
         SUM(CASE WHEN v.is_chinese_owned = 1 THEN 1 ELSE 0 END)                    AS cn_brands_listed,
         SUM(CASE WHEN v.is_chinese_owned = 1 THEN v.registrations_month ELSE 0 END) AS cn_registrations,
         SUM(CASE WHEN v.is_byd = 1           THEN v.registrations_month END)        AS byd_registrations,
         SUM(CASE WHEN v.is_residual = 1      THEN v.registrations_month END)        AS autres_registrations,
         MAX(v.market_registrations_month)                                          AS market_registrations
  FROM v_brand_month v
  GROUP BY v.date_key, v.period, v.year, v.month
) a;

-- ---------------------------------------------------------------------
-- v_chinese_owned_period: Chinese-owned share per full year / Jan-Oct
-- Note: a brand listed earlier in the year but not at the period end has
-- no published YTD and is excluded (Skyworth, Jan-Oct 2026: 74 cars).
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS v_chinese_owned_period;
CREATE VIEW v_chinese_owned_period AS
SELECT p.period_type,
       p.period_label,
       p.year,
       SUM(CASE WHEN p.is_chinese_owned = 1 THEN 1 ELSE 0 END)                         AS cn_brands_counted,
       SUM(CASE WHEN p.is_chinese_owned = 1 THEN p.registrations ELSE 0 END)           AS cn_registrations,
       MAX(p.market_registrations)                                                    AS market_registrations,
       SUM(CASE WHEN p.is_chinese_owned = 1 THEN p.registrations ELSE 0 END) * 100.0
         / MAX(p.market_registrations)                                                AS cn_share_lower_pct,
       SUM(CASE WHEN p.is_chinese_owned = 1 AND p.is_chinese_origin = 1 THEN p.registrations ELSE 0 END) * 100.0
         / MAX(p.market_registrations)                                                AS cn_origin_share_pct,
       (SUM(CASE WHEN p.is_chinese_owned = 1 THEN p.registrations ELSE 0 END)
        + SUM(CASE WHEN p.is_residual = 1 THEN p.registrations ELSE 0 END)) * 100.0
         / MAX(p.market_registrations)                                                AS cn_share_upper_pct,
       SUM(CASE WHEN p.is_byd = 1 THEN p.registrations END) * 100.0
         / SUM(CASE WHEN p.is_chinese_owned = 1 THEN p.registrations ELSE 0 END)      AS byd_part_of_cn_pct
FROM v_brand_period p
GROUP BY p.period_type, p.period_label, p.year;


Writing 04_views.sql


In [ ]:
%%writefile 05_business_queries.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 05_business_queries.sql  (MySQL 8.0)
-- Business questions answered in SQL, on top of 04_views.sql.
-- Each query is preceded by a marker comment (@query, an id, a question) so the
-- notebook can run and display them one by one.
-- Descriptive only: registrations, not retail sales; no causes asserted.
-- =====================================================================

USE byd_france;

-- @query Q1.1 | How is BYD performing? Registrations, share, rank and growth by period
SELECT period_label,
       registrations,
       ROUND(share_pct, 2)                       AS share_pct,
       CONCAT(rank_in_period, ' of ', brands_ranked) AS rank_in_market,
       registrations_prev_year,
       ROUND(growth_pct, 1)                      AS growth_vs_prev_year_pct
FROM v_brand_period
WHERE brand = 'BYD'
ORDER BY period_type, year;

-- @query Q1.2 | BYD's five strongest months by market share
SELECT period,
       registrations_month,
       ROUND(share_month_pct, 2)                 AS share_month_pct,
       CONCAT(rank_month, ' of ', brands_ranked) AS rank_in_month
FROM v_brand_rank_month
WHERE brand = 'BYD'
ORDER BY share_month_pct DESC
LIMIT 5;

-- @query Q3.1 | BYD growth month by month: 2026 vs the same month of 2025
SELECT cur.period,
       cur.registrations_month                   AS registrations_2026,
       prv.registrations_month                   AS registrations_2025,
       ROUND(cur.registrations_month * 100.0 / NULLIF(prv.registrations_month, 0) - 100, 1) AS byd_yoy_pct,
       ROUND(mk.market_yoy_pct, 1)               AS market_yoy_pct
FROM v_brand_month cur
LEFT JOIN v_brand_month prv
       ON prv.brand = cur.brand AND prv.year = cur.year - 1 AND prv.month = cur.month
JOIN v_market_month mk ON mk.date_key = cur.date_key
WHERE cur.brand = 'BYD' AND cur.year = 2026
ORDER BY cur.month;

-- @query Q3.2 | BYD cumulative 2026: running total of monthly figures vs CCFA published YTD
SELECT period,
       registrations_month,
       SUM(registrations_month) OVER (PARTITION BY year ORDER BY month) AS running_total,
       registrations_ytd                                                AS published_ytd,
       registrations_ytd
         - SUM(registrations_month) OVER (PARTITION BY year ORDER BY month) AS ccfa_revision_effect
FROM v_brand_month
WHERE brand = 'BYD' AND year = 2026
ORDER BY month;

-- @query Q4.1 | BYD vs competitors, like-for-like January-October
SELECT p.brand,
       MAX(CASE WHEN p.year = 2024 THEN p.registrations END) AS jan_oct_2024,
       MAX(CASE WHEN p.year = 2025 THEN p.registrations END) AS jan_oct_2025,
       MAX(CASE WHEN p.year = 2026 THEN p.registrations END) AS jan_oct_2026,
       ROUND(MAX(CASE WHEN p.year = 2026 THEN p.growth_pct END), 1)    AS growth_2026_pct,
       ROUND(MAX(CASE WHEN p.year = 2026 THEN p.share_pct END), 2)     AS share_2026_pct,
       MAX(CASE WHEN p.year = 2026 THEN p.rank_in_period END)          AS rank_2026
FROM v_brand_period p
WHERE p.period_type = 'jan_oct'
  AND p.brand IN ('BYD', 'Tesla', 'MG', 'Xpeng', 'Leapmotor')
GROUP BY p.brand
ORDER BY jan_oct_2026 DESC;

-- @query Q4.2 | Head to head in 2026: months in which BYD out-registered Tesla and MG
SELECT b.period,
       b.registrations_month                       AS byd,
       t.registrations_month                       AS tesla,
       g.registrations_month                       AS mg,
       CASE WHEN b.registrations_month > t.registrations_month THEN 'yes' ELSE 'no' END AS byd_above_tesla,
       CASE WHEN b.registrations_month > g.registrations_month THEN 'yes' ELSE 'no' END AS byd_above_mg
FROM v_brand_month b
JOIN v_brand_month t ON t.date_key = b.date_key AND t.brand = 'Tesla'
JOIN v_brand_month g ON g.date_key = b.date_key AND g.brand = 'MG'
WHERE b.brand = 'BYD' AND b.year = 2026
ORDER BY b.period;

-- @query Q4.3 | Competitive positioning: brands with >= 5 000 registrations in Jan-Oct 2025, ranked by growth
WITH mkt AS (
  SELECT MAX(CASE WHEN year = 2026 THEN market_registrations END) * 100.0
         / MAX(CASE WHEN year = 2025 THEN market_registrations END) - 100 AS market_growth_pct
  FROM v_brand_period WHERE period_type = 'jan_oct'
)
SELECT p.brand,
       p.registrations_prev_year                    AS jan_oct_2025,
       p.registrations                              AS jan_oct_2026,
       ROUND(p.growth_pct, 1)                       AS growth_pct,
       ROUND(p.share_pct, 2)                        AS share_2026_pct,
       CASE WHEN p.growth_pct > mkt.market_growth_pct THEN 'faster' ELSE 'slower' END AS vs_market,
       ROUND(mkt.market_growth_pct, 2)              AS market_growth_pct
FROM v_brand_period p
CROSS JOIN mkt
WHERE p.period_type = 'jan_oct' AND p.year = 2026
  AND p.is_residual = 0
  AND p.registrations_prev_year >= 5000
ORDER BY p.growth_pct DESC
LIMIT 10;

-- @query Q5.1 | Chinese-owned brands: combined share by period (lower bound, upper bound incl. AUTRES)
SELECT period_label,
       cn_brands_counted,
       cn_registrations,
       ROUND(cn_share_lower_pct, 2)   AS cn_share_lower_pct,
       ROUND(cn_share_upper_pct, 2)   AS cn_share_upper_pct,
       ROUND(cn_origin_share_pct, 2)  AS cn_origin_only_pct,
       ROUND(byd_part_of_cn_pct, 1)   AS byd_part_of_cn_pct
FROM v_chinese_owned_period
ORDER BY period_type, year;

-- @query Q5.2 | Chinese-owned share by month in 2026, with the AUTRES uncertainty band
SELECT period,
       cn_brands_listed,
       cn_registrations,
       ROUND(cn_share_lower_pct, 2)   AS lower_bound_pct,
       ROUND(cn_share_upper_pct, 2)   AS upper_bound_pct,
       ROUND(byd_part_of_cn_pct, 1)   AS byd_part_of_cn_pct
FROM v_chinese_owned_month
WHERE year = 2026
ORDER BY period;

-- @query Q5.3 | Largest share gains, Jan-Oct 2026 vs Jan-Oct 2025 (percentage points)
SELECT c.brand,
       CASE WHEN c.is_chinese_owned = 1 THEN 'yes' ELSE 'no' END AS chinese_owned,
       ROUND(p.share_pct, 2)                         AS share_2025_pct,
       ROUND(c.share_pct, 2)                         AS share_2026_pct,
       ROUND(c.share_pct - p.share_pct, 2)           AS change_pp
FROM v_brand_period c
JOIN v_brand_period p
  ON p.brand = c.brand AND p.period_type = c.period_type AND p.year = c.year - 1
WHERE c.period_type = 'jan_oct' AND c.year = 2026 AND c.is_residual = 0
ORDER BY change_pp DESC
LIMIT 10;


Writing 05_business_queries.sql


In [ ]:
%%writefile 06_view_checks.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 06_view_checks.sql       (MySQL 8.0)
-- Verifies the views against the figures validated in Notebook 03.
-- Every row must return 'PASS'. Tolerance covers rounding only.
--   mysql -t byd_france < 06_view_checks.sql
-- =====================================================================

USE byd_france;

WITH checks AS (
  SELECT 'V01' AS id, 'BYD growth Jan-Oct 2026 vs 2025 (%)' AS check_name, 185.3 AS expected, 0.05 AS tolerance,
         (SELECT growth_pct FROM v_brand_period WHERE brand = 'BYD' AND period_type = 'jan_oct' AND year = 2026) AS actual
  UNION ALL SELECT 'V02', 'BYD growth FY 2025 vs FY 2024 (%)', 149.9, 0.05,
         (SELECT growth_pct FROM v_brand_period WHERE brand = 'BYD' AND period_type = 'full_year' AND year = 2025)
  UNION ALL SELECT 'V03', 'BYD share Jan-Oct 2026 (%)', 2.03, 0.005,
         (SELECT share_pct FROM v_brand_period WHERE brand = 'BYD' AND period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V04', 'BYD rank Jan-Oct 2026', 16, 0,
         (SELECT rank_in_period FROM v_brand_period WHERE brand = 'BYD' AND period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V05', 'Brands ranked Jan-Oct 2026 (AUTRES excluded)', 44, 0,
         (SELECT brands_ranked FROM v_brand_period WHERE brand = 'BYD' AND period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V06', 'BYD rank Jan-Oct 2024', 32, 0,
         (SELECT rank_in_period FROM v_brand_period WHERE brand = 'BYD' AND period_type = 'jan_oct' AND year = 2024)
  UNION ALL SELECT 'V07', 'BYD monthly rank Jan 2024', 29, 0,
         (SELECT rank_month FROM v_brand_rank_month WHERE brand = 'BYD' AND period = '2024-01')
  UNION ALL SELECT 'V08', 'BYD monthly rank Oct 2026', 7, 0,
         (SELECT rank_month FROM v_brand_rank_month WHERE brand = 'BYD' AND period = '2026-10')
  UNION ALL SELECT 'V09', 'BYD monthly share Oct 2026 (%)', 3.63, 0.005,
         (SELECT share_month_pct FROM v_brand_month WHERE brand = 'BYD' AND period = '2026-10')
  UNION ALL SELECT 'V10', 'Market growth Jan-Oct 2026 vs 2025 (%)', 4.37, 0.005,
         (SELECT market_registrations_ytd * 100.0
                 / (SELECT market_registrations_ytd FROM v_market_month WHERE period = '2025-10') - 100
          FROM v_market_month WHERE period = '2026-10')
  UNION ALL SELECT 'V11', 'August seasonality index (2024-25 average)', 62, 0.5,
         (SELECT AVG(seasonality_index) FROM v_market_month WHERE month = 8 AND is_complete_year = 1)
  UNION ALL SELECT 'V12', 'Tesla growth Jan-Oct 2026 (%)', 117.1, 0.05,
         (SELECT growth_pct FROM v_brand_period WHERE brand = 'Tesla' AND period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V13', 'MG growth Jan-Oct 2026 (%)', 23.7, 0.05,
         (SELECT growth_pct FROM v_brand_period WHERE brand = 'MG' AND period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V14', 'Leapmotor growth Jan-Oct 2026 (%)', 172.6, 0.05,
         (SELECT growth_pct FROM v_brand_period WHERE brand = 'Leapmotor' AND period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V15', 'Xpeng growth Jan-Oct 2026 (%)', 154.1, 0.05,
         (SELECT growth_pct FROM v_brand_period WHERE brand = 'Xpeng' AND period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V16', 'Xpeng Jan-Oct 2025 growth is NULL (no 2024 figure; 1 = yes)', 1, 0,
         (SELECT CASE WHEN growth_pct IS NULL THEN 1 ELSE 0 END FROM v_brand_period
          WHERE brand = 'Xpeng' AND period_type = 'jan_oct' AND year = 2025)
  UNION ALL SELECT 'V17', 'Chinese-owned share FY 2024 (%)', 2.95, 0.005,
         (SELECT cn_share_lower_pct FROM v_chinese_owned_period WHERE period_type = 'full_year' AND year = 2024)
  UNION ALL SELECT 'V18', 'Chinese-owned share FY 2025 (%)', 4.20, 0.005,
         (SELECT cn_share_lower_pct FROM v_chinese_owned_period WHERE period_type = 'full_year' AND year = 2025)
  UNION ALL SELECT 'V19', 'Chinese-owned share Jan-Oct 2026 (%)', 7.07, 0.005,
         (SELECT cn_share_lower_pct FROM v_chinese_owned_period WHERE period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V20', 'Chinese-owned brands counted Jan-Oct 2026', 11, 0,
         (SELECT cn_brands_counted FROM v_chinese_owned_period WHERE period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V21', 'Chinese-origin only share Jan-Oct 2026 (%)', 4.21, 0.005,
         (SELECT cn_origin_share_pct FROM v_chinese_owned_period WHERE period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V22', 'BYD part of Chinese-owned volume FY 2024 (%)', 10.7, 0.05,
         (SELECT byd_part_of_cn_pct FROM v_chinese_owned_period WHERE period_type = 'full_year' AND year = 2024)
  UNION ALL SELECT 'V23', 'BYD part of Chinese-owned volume Jan-Oct 2026 (%)', 28.7, 0.05,
         (SELECT byd_part_of_cn_pct FROM v_chinese_owned_period WHERE period_type = 'jan_oct' AND year = 2026)
  UNION ALL SELECT 'V24', 'Months where lower bound > upper bound', 0, 0,
         (SELECT COUNT(*) FROM v_chinese_owned_month WHERE cn_share_lower_pct > cn_share_upper_pct)
  UNION ALL SELECT 'V25', 'Months where upper - lower <> AUTRES share', 0, 0,
         (SELECT COUNT(*) FROM v_chinese_owned_month
          WHERE ABS(cn_share_upper_pct - cn_share_lower_pct - autres_share_pct) > 0.000001 OR autres_share_pct IS NULL)
  UNION ALL SELECT 'V26', 'Brands with >= 5 000 regs Jan-Oct 2025', 28, 0,
         (SELECT COUNT(*) FROM v_brand_period WHERE period_type = 'jan_oct' AND year = 2026
          AND is_residual = 0 AND registrations_prev_year >= 5000)
  UNION ALL SELECT 'V27', 'Monthly ranking rows with missing registrations', 0, 0,
         (SELECT COUNT(*) FROM v_brand_rank_month WHERE registrations_month IS NULL)
  UNION ALL SELECT 'V28', 'Universe rows in v_brand_month (in_market_sum = 1, incl. AUTRES)', 1408, 0,
         (SELECT COUNT(*) FROM v_brand_month)
)
SELECT id, check_name, expected, ROUND(actual, 3) AS actual,
       CASE WHEN actual IS NOT NULL AND ABS(actual - expected) <= tolerance THEN 'PASS' ELSE 'FAIL' END AS result
FROM checks
ORDER BY id;


Writing 06_view_checks.sql


## 4 · Rebuild and reload the model, then re-run the 22 load checks
Same steps as Notebook 04. **Expected:** 0 load warnings, and all 22 checks PASS.

In [ ]:
mysql("01_schema.sql", db=None)
loads = pd.concat(mysql("02_load.sql", local_infile=True), ignore_index=True)
display(loads)
assert (loads.warnings == 0).all()
assert dict(zip(loads.loaded_table, loads.rows_loaded)) == {"dim_brand": 51, "fact_market_month": 34, "fact_brand_month": 1734}
load_checks = mysql("03_checks.sql")[0]
print(f"Load checks: {(load_checks.result == 'PASS').sum()} / {len(load_checks)} PASS")
assert (load_checks.result == "PASS").all()

,loaded_table,rows_loaded,warnings
0,dim_brand,51,0
1,fact_market_month,34,0
2,fact_brand_month,1734,0


Load checks: 22 / 22 PASS


## 5 · Create the views
| View | Grain | Used for |
|---|---|---|
| `v_brand_month` | brand × month (universe only) | share, trends; the main Power BI fact |
| `v_market_month` | month | market size, year-on-year change, seasonality index |
| `v_brand_period` | brand × period (full year / Jan–Oct) | totals, share, rank, growth |
| `v_brand_rank_month` | brand × month | monthly rank (AUTRES excluded) |
| `v_chinese_owned_month` | month | Chinese-owned share: lower and upper bound |
| `v_chinese_owned_period` | period | the same, per full year / Jan–Oct |

In [ ]:
mysql("04_views.sql")
mysql(query="SELECT table_name AS view_name FROM information_schema.views WHERE table_schema = 'byd_france' ORDER BY table_name")[0]

,view_name
0,v_brand_month
1,v_brand_period
2,v_brand_rank_month
3,v_chinese_owned_month
4,v_chinese_owned_period
5,v_market_month


## 6 · Check the views against Notebook 03
28 checks. Each recomputes a validated figure from the views: BYD growth, share and ranks; competitor growth; market growth and seasonality; Chinese-owned shares and bounds; NULL handling (Xpeng has no 2024 figure, so its 2025 growth must be NULL). **Every row must say PASS.**

In [ ]:
view_checks = mysql("06_view_checks.sql")[0]
display(view_checks)
print("ALL VIEW CHECKS PASS:", bool((view_checks.result == "PASS").all()))
assert (view_checks.result == "PASS").all()

,id,check_name,expected,actual,result
0,V01,BYD growth Jan-Oct 2026 vs 2025 (%),185.30,185.340,PASS
1,V02,BYD growth FY 2025 vs FY 2024 (%),149.90,149.917,PASS
2,V03,BYD share Jan-Oct 2026 (%),2.03,2.029,PASS
3,V04,BYD rank Jan-Oct 2026,16.00,16.000,PASS
4,V05,Brands ranked Jan-Oct 2026 (AUTRES excluded),44.00,44.000,PASS
5,V06,BYD rank Jan-Oct 2024,32.00,32.000,PASS
6,V07,BYD monthly rank Jan 2024,29.00,29.000,PASS
7,V08,BYD monthly rank Oct 2026,7.00,7.000,PASS
8,V09,BYD monthly share Oct 2026 (%),3.63,3.629,PASS
9,V10,Market growth Jan-Oct 2026 vs 2025 (%),4.37,4.372,PASS


ALL VIEW CHECKS PASS: True


## 7 · Business queries
Each query below runs from `05_business_queries.sql`. The tables are descriptive: they show registrations, not retail sales, and not the causes behind them.

In [ ]:
queries = re.findall(r"^-- @query (Q[\d.]+) \| (.*?)\n(.*?)(?=^-- @query|\Z)",
                     open("05_business_queries.sql").read(), re.S | re.M)
print(f"{len(queries)} queries found")
results = {}
def run_query(qid):
    title, sql_text = next((t, s) for q, t, s in queries if q == qid)
    results[qid] = mysql(query=sql_text)[0]
    display(Markdown(f"**{qid} · {title}**"))
    display(results[qid])

10 queries found


### Q1 · How is BYD performing in France?

In [ ]:
run_query("Q1.1")
run_query("Q1.2")

**Q1.1 · How is BYD performing? Registrations, share, rank and growth by period**

,period_label,registrations,share_pct,rank_in_market,registrations_prev_year,growth_vs_prev_year_pct
0,FY 2024,5415,0.32,30 of 40,NaN,NaN
1,FY 2025,13533,0.83,24 of 40,5415.0,149.9
2,Jan-Oct 2024,2968,0.21,32 of 40,NaN,NaN
3,Jan-Oct 2025,9843,0.74,24 of 40,2968.0,231.6
4,Jan-Oct 2026,28086,2.03,16 of 44,9843.0,185.3


**Q1.2 · BYD's five strongest months by market share**

,period,registrations_month,share_month_pct,rank_in_month
0,2026-10,5419,3.63,7 of 44
1,2026-06,5040,2.67,11 of 42
2,2026-09,3972,2.54,11 of 42
3,2026-07,2954,2.33,13 of 42
4,2026-08,2195,2.33,13 of 42


**Reading it:** compare like with like. Full years (2024, 2025) are compared with each other, and January–October with January–October. BYD's five strongest share months all fall in 2026. **Growth is NULL** for the first year of each series, because there is no earlier figure to compare with.

### Q3 · How is BYD growing over time?

In [ ]:
run_query("Q3.1")
run_query("Q3.2")

**Q3.1 · BYD growth month by month: 2026 vs the same month of 2025**

,period,registrations_2026,registrations_2025,byd_yoy_pct,market_yoy_pct
0,2026-01,872,366,138.3,-6.6
1,2026-02,976,602,62.1,-14.7
2,2026-03,1540,659,133.7,12.9
3,2026-04,2533,2064,22.7,-0.3
4,2026-05,2585,938,175.6,3.7
5,2026-06,5040,1011,398.5,11.4
6,2026-07,2954,805,267.0,9.0
7,2026-08,2195,1338,64.1,7.4
8,2026-09,3972,1040,281.9,11.6
9,2026-10,5419,1020,431.3,7.0


**Q3.2 · BYD cumulative 2026: running total of monthly figures vs CCFA published YTD**

,period,registrations_month,running_total,published_ytd,ccfa_revision_effect
0,2026-01,872,872,872,0
1,2026-02,976,1848,1848,0
2,2026-03,1540,3388,3388,0
3,2026-04,2533,5921,5921,0
4,2026-05,2585,8506,8506,0
5,2026-06,5040,13546,13546,0
6,2026-07,2954,16500,16500,0
7,2026-08,2195,18695,18695,0
8,2026-09,3972,22667,22667,0
9,2026-10,5419,28086,28086,0


**Reading it:** Q3.1 compares each 2026 month with the same month of 2025, so seasonality cancels out, with the market's own change alongside. Q3.2 is a **running total** (window function). `ccfa_revision_effect` shows whether CCFA's published YTD differs from simply adding up the months; a non-zero value would mean CCFA revised earlier months.

### Q4 · How does BYD compare with competitors?

In [ ]:
run_query("Q4.1")
run_query("Q4.2")
run_query("Q4.3")

**Q4.1 · BYD vs competitors, like-for-like January-October**

,brand,jan_oct_2024,jan_oct_2025,jan_oct_2026,growth_2026_pct,share_2026_pct,rank_2026
0,Tesla,31217.0,21927,47609,117.1,3.44,7
1,MG,15151.0,23872,29524,23.7,2.13,14
2,BYD,2968.0,9843,28086,185.3,2.03,16
3,Leapmotor,252.0,2675,7292,172.6,0.53,28
4,Xpeng,NaN,2461,6254,154.1,0.45,29


**Q4.2 · Head to head in 2026: months in which BYD out-registered Tesla and MG**

,period,byd,tesla,mg,byd_above_tesla,byd_above_mg
0,2026-01,872,661,1129,yes,no
1,2026-02,976,3715,2136,no,no
2,2026-03,1540,9569,2864,no,no
3,2026-04,2533,1829,2918,yes,no
4,2026-05,2585,5446,2682,no,no
5,2026-06,5040,7474,4688,no,yes
6,2026-07,2954,2429,3343,yes,no
7,2026-08,2195,5038,2003,no,yes
8,2026-09,3972,9038,3787,no,yes
9,2026-10,5419,2410,3974,yes,yes


**Q4.3 · Competitive positioning: brands with >= 5 000 registrations in Jan-Oct 2025, ranked by growth**

,brand,jan_oct_2025,jan_oct_2026,growth_pct,share_2026_pct,vs_market,market_growth_pct
0,BYD,9843,28086,185.3,2.03,faster,4.37
1,Tesla,21927,47609,117.1,3.44,faster,4.37
2,Fiat + Abarth,16998,27099,59.4,1.96,faster,4.37
3,MG,23872,29524,23.7,2.13,faster,4.37
4,Volvo,9197,10108,9.9,0.73,faster,4.37
5,Citroën,95802,105070,9.7,7.59,faster,4.37
6,Skoda,40591,44115,8.7,3.19,faster,4.37
7,Kia,27535,29898,8.6,2.16,faster,4.37
8,Renault,230643,238448,3.4,17.23,slower,4.37
9,Mini,20866,21064,0.9,1.52,slower,4.37


**Reading it:** Q4.1 pivots years into columns with conditional aggregation. Xpeng's 2024 cell is NULL: it wasn't listed separately, so it is unknown, not zero. Q4.2 shows month-by-month head-to-heads. Q4.3 applies the same ≥ 5 000 base rule as Notebook 03, so small brands can't top the list on percentage growth alone.

### Q5 · What opportunities exist? Chinese-owned brands

In [ ]:
run_query("Q5.1")
run_query("Q5.2")
run_query("Q5.3")

**Q5.1 · Chinese-owned brands: combined share by period (lower bound, upper bound incl. AUTRES)**

,period_label,cn_brands_counted,cn_registrations,cn_share_lower_pct,cn_share_upper_pct,cn_origin_only_pct,byd_part_of_cn_pct
0,FY 2024,5,50656,2.95,3.06,0.36,10.7
1,FY 2025,7,68568,4.20,4.31,1.33,19.7
2,Jan-Oct 2024,5,33974,2.42,2.52,0.27,8.7
3,Jan-Oct 2025,6,48717,3.67,3.80,1.18,20.2
4,Jan-Oct 2026,11,97896,7.07,7.20,4.21,28.7


**Q5.2 · Chinese-owned share by month in 2026, with the AUTRES uncertainty band**

,period,cn_brands_listed,cn_registrations,lower_bound_pct,upper_bound_pct,byd_part_of_cn_pct
0,2026-01,7,3334,3.11,3.19,26.2
1,2026-02,7,4656,3.86,3.96,21.0
2,2026-03,7,6584,3.79,3.88,23.4
3,2026-04,7,7525,5.44,6.34,33.7
4,2026-05,9,8415,6.55,6.66,30.7
5,2026-06,9,14961,7.92,8.12,33.7
6,2026-07,9,11064,8.73,9.02,26.7
7,2026-08,9,7696,8.16,8.51,28.5
8,2026-09,9,14168,9.05,9.74,28.0
9,2026-10,11,17220,11.53,11.66,31.5


**Q5.3 · Largest share gains, Jan-Oct 2026 vs Jan-Oct 2025 (percentage points)**

,brand,chinese_owned,share_2025_pct,share_2026_pct,change_pp
0,Tesla,no,1.65,3.44,1.79
1,BYD,yes,0.74,2.03,1.29
2,Fiat + Abarth,no,1.28,1.96,0.68
3,Citroën,no,7.22,7.59,0.37
4,MG,yes,1.80,2.13,0.33
5,Leapmotor,yes,0.20,0.53,0.33
6,Xpeng,yes,0.19,0.45,0.27
7,Skoda,no,3.06,3.19,0.13
8,Alpine,no,0.32,0.43,0.12
9,Kia,no,2.08,2.16,0.08


**Reading it:** the Chinese-owned share is a **range**. The lower bound counts only brands CCFA lists separately; the upper bound adds the whole AUTRES residual, which may contain unlisted Chinese brands. Part of the rise comes from **more brands being listed** (see `cn_brands_listed`). Q5.3 includes only brands with a figure in both years.

## 8 · Final validation

In [ ]:
final = {
    "Input files match validated v2 (SHA-256)": all(hashlib.sha256(open(n, "rb").read()).hexdigest() == s for n, s in EXPECTED_SHA256.items()),
    "Load: 0 warnings, expected row counts": bool((loads.warnings == 0).all()),
    "22 load checks PASS": bool((load_checks.result == "PASS").all()),
    "6 views created": len(mysql(query="SELECT table_name FROM information_schema.views WHERE table_schema='byd_france'")[0]) == 6,
    "28 view checks PASS": bool((view_checks.result == "PASS").all()) and len(view_checks) == 28,
    "10 business queries returned rows": len(results) == 10 and all(len(df) > 0 for df in results.values()),
    "BYD Jan-Oct 2026 growth = 185.3% (Q1.1)": float(results["Q1.1"].set_index("period_label").loc["Jan-Oct 2026", "growth_vs_prev_year_pct"]) == 185.3,
    "Xpeng 2024 is NULL, not 0 (Q4.1)": bool(pd.isna(results["Q4.1"].set_index("brand").loc["Xpeng", "jan_oct_2024"])),
}
for k, v in final.items():
    print(f"{k:<45} {v}")
assert all(final.values())

Input files match validated v2 (SHA-256)      True
Load: 0 warnings, expected row counts         True
22 load checks PASS                           True
6 views created                               True
28 view checks PASS                           True
10 business queries returned rows             True
BYD Jan-Oct 2026 growth = 185.3% (Q1.1)       True
Xpeng 2024 is NULL, not 0 (Q4.1)              True


## Summary
* **Six views** now hold the project's business logic (universe, share, period totals, ranks, growth, Chinese-owned bounds) in one place.
* **28 view checks pass**: the SQL results equal the validated Notebook 03 figures.
* **10 business queries** answer Q1, Q3, Q4 and Q5 in SQL with CTEs, window functions, self-joins and conditional aggregation, with missing values kept as NULL throughout.

**Next:** Phase 5, KPI definitions (`docs/kpi_definitions.md`), then the Power BI model built on these views.